In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types
import os 

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [3]:
spark.version

'4.1.1'

In [6]:
#Question 2
df_yellow = spark.read \
        .parquet("yellow_tripdata_2025-11.parquet")

df_yellow \
        .repartition(4) \
        .write.parquet('homeworkdata/', mode='overwrite')

In [12]:
mean = 0
for file in os.listdir('homeworkdata/'):
    if file.endswith('.parquet'):
        full_path = os.path.join('homeworkdata/', file)
        size = os.path.getsize(full_path) / (1024**2)
        mean += size
        print(f"{file}: {size:.2f} MB")
print(f"Average file size: {mean/4:.2f} MB")

part-00000-2fb7b2dd-f317-4e61-a00a-f7b05288c202-c000.snappy.parquet: 24.39 MB
part-00001-2fb7b2dd-f317-4e61-a00a-f7b05288c202-c000.snappy.parquet: 24.42 MB
part-00002-2fb7b2dd-f317-4e61-a00a-f7b05288c202-c000.snappy.parquet: 24.42 MB
part-00003-2fb7b2dd-f317-4e61-a00a-f7b05288c202-c000.snappy.parquet: 24.41 MB
Average file size: 24.41 MB


In [14]:
df_yellow.createOrReplaceTempView('yellow')

In [24]:
#df_yellow.columns
df_yellow.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [27]:
spark.sql("""
SELECT
COUNT(*)
FROM yellow
WHERE YEAR(tpep_pickup_datetime)=2025 
      AND MONTH(tpep_pickup_datetime)=11 
      AND DAY(tpep_pickup_datetime)=15
        """         
          ).show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [36]:
spark.sql("""
SELECT *
FROM
(SELECT *, (unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600 AS trip_duration_hours
FROM yellow
)A order by trip_duration_hours DESC
        """         
          ).show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|trip_duration_hours|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-------------------+
|       2| 2025-11-26 20:22:12|  2025-11-30 15:01:00|              1| 

In [47]:
df_zones = spark.read \
        .option("header", "true") \
        .csv("taxi_zone_lookup.csv") \
        .createOrReplaceTempView('zones')

In [50]:
spark.sql("""
    SELECT COUNT(*) as trip_count, z.Zone
    FROM yellow y
    LEFT JOIN zones z ON y.PULocationID = z.LocationID  
    GROUP BY z.Zone   
    ORDER BY trip_count  ASC         
        """).show(5)

+----------+--------------------+
|trip_count|                Zone|
+----------+--------------------+
|         1|Governor's Island...|
|         1|Eltingville/Annad...|
|         1|       Arden Heights|
|         3|       Port Richmond|
|         4|       Rikers Island|
+----------+--------------------+
only showing top 5 rows
